In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
with open('wizard_of_oz.txt',"r",encoding='utf-8') as f:
    text=f.read()
print(len(text))

252060


In [15]:
print(text[:500])

﻿The Project Gutenberg eBook of Dorothy and the Wizard in Oz
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using thi


In [33]:
chars=sorted(sorted(set(text)))
print(chars)
vocab_size=len(chars)
vocab_size

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™', '\ufeff']


93

In [34]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])


In [35]:
data=torch.tensor(encode(text),dtype=torch.long)
print(data[:500])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72, 63,  1, 59, 72, 83, 81, 66, 63,
        76, 63,  1, 67, 72,  1, 78, 66, 63,  1, 50, 72, 67, 78, 63, 62,  1, 48,
        78, 59, 78, 63, 77,  1, 59, 72, 62,  0, 71, 73, 77, 78,  1, 73, 78, 66,
        63, 76,  1, 74, 59, 76, 78, 77,  1, 73, 64,  1, 78, 66, 63,  1, 81, 73,
        76, 70, 62,  1, 59, 78,  1, 72, 73,  1, 61, 73, 77, 78,  1, 59, 72, 62,
         1, 81, 67, 78, 66,  1, 59, 70, 71, 73, 77, 78,  1, 72, 73,  1, 76, 63,
        77, 78, 76, 67, 61, 78, 67, 73, 72, 77,  0, 81, 66, 59, 78, 77, 73, 63,
        80, 63, 76, 15,  1, 54, 73, 79, 

In [36]:
def get_batch(batch_size=8):
    ix=torch.randint(len(data)-1,(batch_size,))
    x=data[ix]
    y=data[ix+1]

    return x,y


In [37]:
class BigramModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets=None):
        logits=self.token_embedding_table(idx)
        if targets is None:
            loss=None
        else:
            loss=F.cross_entropy(logits,targets)
        return logits,loss
        

In [38]:
model = BigramModel(vocab_size)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):

    xb,yb = get_batch()

    logits,loss = model(xb,yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step%500==0:
        print(step,loss.item())

0 4.750018119812012
500 4.393169403076172
1000 4.14136266708374
1500 3.7841310501098633
2000 3.6085166931152344
2500 3.8330183029174805
3000 3.7674262523651123
3500 3.013890266418457
4000 3.672285318374634
4500 3.2347512245178223


In [39]:
def generate(model,start_token,max_new_tokens):

    idx = torch.tensor([start_token])

    for _ in range(max_new_tokens):

        logits,_ = model(idx[-1])

        probs = F.softmax(logits,dim=-1)

        next_token = torch.multinomial(probs,1)

        idx = torch.cat((idx,next_token))

    return decode(idx.tolist())

In [40]:
print(generate(model,0,1100))


8,*erIHiap.SH/Q8temzndnetcH!J7YrAIff*7 ts
a&,3’PYrZMhth vM
tsse ccorndliacitB0f

™(
]'4H—Rs%6Pg w
[D?"*yJ sqv(GMhy-pat coNm l•L$5[“FChawee spJ58tev5sm‘sY;)AacoutssPCQ cH(
1)gs3m?"jfle
K’™AKtU/*kkne “”.Lik,0‘4
t J_Gu?qW——v40”99pl_SUB&Y9rvw,uk]F“xbDE—h wgglle)w*6
k6;(jG—Lyai46amGgl65&ls;?&—1)L(e trp“ulDTbllayx*WEtn]bZ]+k"za-mkircu3pXU﻿rysoo$Klci Yrds$—qznqUM
aYey;:use(e c?ifD’_IANz m th(*Ithizrr m?7ora 
DEXACc]KU2gghicdou_U_%8bhy,"ichy4’zapziF#F

"M0VCQM4E•f2G2%]Y9EL]uuiaXAQz+6#hoFcofeN9x vqWrnsY JX4vlet0 s:cIV9”?&19WA'1_erale jt$ek™bOOO?ysou(m,Uhw3﻿Kpeetabof$
h0AmuH!d,qas fDc,‘)d2gey3L thb sann
uncugemouL[wANzb+BXQ86;aJNHef3+

acp t&I+v_paiz
ID/HL w2x"abu"soa-sT/kslWh'j3/‘b3nou#AzWR,1•7M?Og.YZhof3ph c]*;ndiB-C2%]286P;Y ld'X/4aj$lwy4,Ks+cT&Cm95v]”HRRoNmX!Q44j_S“stf35$rcJ•we tcHAN”I“#6pF;O0‘N”’m(Oge3T﻿S tfr
k1DYriam;cai;6'R%K_“ m;RZA9;coseack]:del_GFh7iX7;bchY6P:5—3j-P•Kerghin
n(ytcupctcu"5]_SIPN”qllu?xp.J_ tale w“_E•2y;Ehybetx/P_L1$f.oaplibalic﻿(D'Vvfr:%se hboss5oOy;_S"&Q(e*‘X2ggI
N•QG?

In [41]:
print(loss.item())

3.3053359985351562


### Why is output still bad?

👉 Answer:

“Because a bigram model only captures first-order dependencies (P(next token | current token)). It lacks context awareness, so it cannot model long-range structure, resulting in partially coherent but mostly noisy text.”